# Matching MGCP polygon features to Overture

This notebook demonstrates a methodology for matching polygon features from
the [Multinational Geospatial Co-Production Program (MGCP)](https://www.dgiwg.org/dgiwg-standards)
to features in Overture data, and producing a stable [GERS](https://docs.overturemaps.org/gers/)
link table that can serve as the join key for downstream data integration.

The demo uses publicly available MGCP TRD 3.0 data over the Bahamas as the
input, and matches it against six Overture types: `buildings/building` and
five polygon-geometry types within the `base` theme. The methodology is
geometry-based — it works the same regardless of which schema version the
input data is captured against — and the same approach applies to other
polygon datasets you might want to align with Overture's reference map.

**What you'll do in this notebook:**

1. Load a single MGCP cell and inventory the polygon features it contains.
2. Pull Overture data for the same geographic extent across six polygon types.
3. Run six matching passes, one MGCP-to-Overture-type pair at a time.
4. Classify the results into cardinality categories (1:1, 1:many, many:1, 1:0, 0:1).
5. Use those results to think about how a dataset like this could be linked
   to GERS as a stable reference identifier.

A short note on scope: the demo matches *polygon* features only. Many smaller
structures in MGCP are captured as points at this scale, and matching those
to Overture requires a different algorithm (point-in-polygon). That's a
natural follow-on but is out of scope here.

## Setup

Imports, paths, and the current Overture release.

In [ ]:
from pathlib import Path

import duckdb
import geopandas as gpd
import pandas as pd
import pystac
from shapely import wkb

# Resolve repo root from notebook location, so the notebook works whether
# Jupyter was launched from the repo root or from notebooks/.
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA = REPO_ROOT / "data"

Overture publishes a STAC catalog of every release. The custom `latest` field
on the catalog points at the current release id, so we don't have to hardcode
a release that will go stale.

In [ ]:
catalog = pystac.Catalog.from_file("https://stac.overturemaps.org/catalog.json")
OVERTURE_RELEASE = catalog.extra_fields["latest"]
print(f"Using Overture release: {OVERTURE_RELEASE}")

## The demo tile

MGCP data is organized into 1-degree by 1-degree cells. For this demo we use
the cell covering longitudes -79 to -78 and latitudes 26 to 27, which sits
over the western Bahamas (including parts of Grand Bahama Island). The cell
is identified by the codename `W079N26`.

**To run this notebook,** download the cell ZIP from ArcGIS Online and
unpack it into `data/mgcp/W079N26/`. The cell is publicly available at:

> https://www.arcgis.com/home/item.html?id=e6f2fc2edb744e3ca9a4b6e60486fcd8

After unpacking you should have a directory containing many `A*.shp`,
`P*.shp`, `L*.shp` shapefile groups — one for each combination of geometry
type and feature code that has at least one feature in this cell.

In [ ]:
MGCP_DIR = DATA / "mgcp" / "W079N26"
TILE_BBOX = (-79, 26, -78, 27)  # (minx, miny, maxx, maxy)

if not MGCP_DIR.exists():
    raise FileNotFoundError(
        f"MGCP cell not found at {MGCP_DIR}. See the cell above for download instructions."
    )

# Count shapefiles by geometry-type prefix
shapefile_groups = sorted(MGCP_DIR.glob("*.shp"))
print(f"Found {len(shapefile_groups)} shapefiles in {MGCP_DIR.name}")
print()
print("By geometry-type prefix:")
prefixes = {"A": "area (polygon)", "P": "point", "L": "line", "T": "text"}
for prefix, label in prefixes.items():
    n = sum(1 for p in shapefile_groups if p.stem.startswith(prefix))
    if n:
        print(f"  {prefix} {label:<20} {n:>4} shapefiles")

## Load MGCP polygon features

Polygon features in MGCP are stored in shapefiles with the `A` prefix
(for "area"). The rest of the filename is the MGCP feature code — for
example, `AAL015.shp` contains polygons for feature code `AL015`, which
is "Building" in MGCP TRD 3.0.

We load every polygon shapefile in the cell into a single GeoDataFrame,
keeping the feature code as a column so we can group on it later. Each
shapefile has its own attribute schema, so we keep only the columns common
to all (UID, FCODE) plus the geometry.

In [ ]:
def load_mgcp_polygons(mgcp_dir):
    """Load all A*.shp polygon shapefiles in an MGCP cell directory.

    Returns a GeoDataFrame with columns: UID, FCODE, geometry.
    All other per-feature attributes are dropped here; they can be joined
    back in later if needed.
    """
    parts = []
    for shp in sorted(mgcp_dir.glob("A*.shp")):
        gdf = gpd.read_file(shp)
        if len(gdf) == 0:
            continue
        # Keep only the identifiers and geometry; the attribute schema
        # varies by feature code and we don't need attributes for matching.
        parts.append(gdf[["UID", "FCODE", "geometry"]])
    if not parts:
        return gpd.GeoDataFrame(
            columns=["UID", "FCODE", "geometry"], geometry="geometry", crs="EPSG:4326"
        )
    out = pd.concat(parts, ignore_index=True)
    return gpd.GeoDataFrame(out, geometry="geometry", crs="EPSG:4326")


mgcp = load_mgcp_polygons(MGCP_DIR)
print(f"Loaded {len(mgcp):,} MGCP polygon features")
print(f"Across {mgcp['FCODE'].nunique()} distinct feature codes")
print(f"\nCRS: {mgcp.crs}")
print(f"Bounding box: {tuple(round(float(v), 4) for v in mgcp.total_bounds)}")

## Inventory by feature code

Which feature types are present in this cell, and how many of each? This
matters for two reasons: it tells us where the matching workload is
concentrated, and it lets us understand which MGCP feature codes will
need to find a home in Overture's schema.

In [ ]:
inventory = (
    mgcp.groupby("FCODE", as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
inventory

A few things to notice:

- **Buildings (`AL015`) are the largest single category** by polygon count,
  but they're not the majority. Most polygons in the cell are vegetation,
  land cover, or hydrography.
- **The feature codes span multiple broad themes:** structures (`AL*`, `AM*`,
  `AD*`), hydrography (`BA*`, `BB*`, `BH*`), vegetation and land cover
  (`EA*`, `EB*`, `EC*`, `ED*`), and a few others. Overture's schema doesn't
  align one-to-one with this; some MGCP codes correspond to Overture
  `buildings`, others to types within the `base` theme.
- **This sparseness is real.** MGCP data captured at 1:100,000 scale doesn't
  try to capture every small structure. Many features that exist in dense
  Overture data simply aren't in the MGCP cell. We'll see this clearly when
  we compare totals.

## Pull Overture data for the demo tile

We're matching MGCP polygons against six Overture types that cover the
range of polygon features in the demo cell:

- `buildings/building` — building footprints
- `base/infrastructure` — utility, communication, and similar structures (polygons only)
- `base/land_use` — how land is used (e.g., residential, industrial)
- `base/land_cover` — what's on the land (e.g., vegetation, bare earth)
- `base/land` — physical land features (e.g., reef, beach, island)
- `base/water` — water bodies (polygons only)

The base theme contains five types in total; we use all of them that carry
polygon geometry. Some types (`infrastructure`, `water`) contain a mix of
geometry types — we filter to polygons only.

Overture data is published as GeoParquet on S3. We use DuckDB's spatial
extension to pull a subset of each type filtered to the demo tile's
bounding box. The first run hits S3; subsequent runs use a local cache.

In [ ]:
OVERTURE_CACHE = DATA / "overture_cache"
OVERTURE_CACHE.mkdir(parents=True, exist_ok=True)

OVERTURE_TYPES = [
    # The clean case: building-to-building matching
    ("buildings", "building"),
    ("buildings", "building_part"),
    # The adjacent cases: similar behavior, smaller volume
    ("base", "infrastructure"),
    ("base", "land_use"),
    ("base", "water"),
    # The cross-schema friction cases: MGCP's specific codes vs Overture's broader taxonomy
    ("base", "land"),
    ("base", "land_cover"),
]


def pull_overture_polygons(theme, type_, bbox, release, cache_dir):
    """Pull Overture polygon features for one theme/type within a bbox.

    Filters to polygon and multipolygon geometries only. Caches the result
    to a local parquet so subsequent runs don't re-hit S3.
    """
    cache_path = cache_dir / f"{theme}_{type_}.parquet"
    if cache_path.exists():
        return cache_path

    s3_path = f"s3://overturemaps-us-west-2/release/{release}/theme={theme}/type={type_}/*"
    minx, miny, maxx, maxy = bbox

    con = duckdb.connect()
    con.execute("INSTALL spatial; LOAD spatial;")
    con.execute("INSTALL httpfs; LOAD httpfs;")
    con.execute("SET s3_region = 'us-west-2';")

    sql = f"""
        SELECT *
        FROM read_parquet('{s3_path}', hive_partitioning=1)
        WHERE bbox.xmin BETWEEN {minx} AND {maxx}
          AND bbox.ymin BETWEEN {miny} AND {maxy}
          AND ST_GeometryType(geometry) IN ('POLYGON', 'MULTIPOLYGON')
    """
    df = con.execute(sql).fetchdf()
    df.to_parquet(cache_path)
    return cache_path


# Pull (or load from cache) each Overture type
overture_paths = {}
for theme, type_ in OVERTURE_TYPES:
    key = f"{theme}/{type_}"
    print(f"Fetching {key}...", end=" ", flush=True)
    path = pull_overture_polygons(theme, type_, TILE_BBOX, OVERTURE_RELEASE, OVERTURE_CACHE)
    overture_paths[key] = path
    print(f"cached at {path.name}")

## Load Overture parquets as GeoDataFrames

The parquets DuckDB wrote contain the geometry as raw WKB bytes rather than
as a GeoParquet-typed column, so we parse the geometry column ourselves
before constructing the GeoDataFrame.

In [ ]:
def load_overture_parquet(path):
    """Load a DuckDB-written parquet as a GeoDataFrame."""
    df = pd.read_parquet(path)
    df["geometry"] = df["geometry"].apply(wkb.loads)
    return gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")


overture = {key: load_overture_parquet(path) for key, path in overture_paths.items()}
for key, gdf in overture.items():
    print(f"{key:<30} {len(gdf):>7,} features")

## Compare MGCP and Overture polygon coverage

Side by side, here's what's in the demo cell from each source. The two
datasets break down naturally along different axes — Overture by theme and
type, MGCP by feature code — so we look at each on its own terms before
comparing.

In [ ]:
# MGCP feature-code names from a curated lookup at mgcp_fcode_names.csv.
# See that file for source attribution per code.
fcode_names = pd.read_csv(REPO_ROOT / "mgcp_fcode_names.csv")

# Overture breakdown by theme/type
overture_breakdown = pd.DataFrame(
    [(key, len(gdf)) for key, gdf in overture.items()],
    columns=["theme/type", "polygons"],
).sort_values("polygons", ascending=False).reset_index(drop=True)

# MGCP breakdown with names joined on
mgcp_breakdown = (
    inventory.rename(columns={"FCODE": "feature_code", "count": "polygons"})
    .merge(fcode_names[["feature_code", "name"]], on="feature_code", how="left")
    .fillna({"name": ""})
    [["feature_code", "name", "polygons"]]
)

print(f"Overture: {sum(len(g) for g in overture.values()):,} polygons across {len(overture)} theme/type combinations")
print(f"MGCP:     {len(mgcp):,} polygons across {mgcp['FCODE'].nunique()} feature codes")
print(f"Ratio:    {sum(len(g) for g in overture.values()) / len(mgcp):.1f}x more Overture polygons\n")

print("Overture by theme/type:")
print(overture_breakdown.to_string(index=False))
print(f"\nMGCP by feature code (top 10 of {len(mgcp_breakdown)}):")
print(mgcp_breakdown.head(10).to_string(index=False))

### A note on density

A 1° × 1° cell is around 12,300 km². Why so few features per square
kilometer? Two reasons stack up: most of this cell is open ocean, and
MGCP's 1:100,000 capture scale doesn't try to record every small structure
the way denser ML-derived datasets do. The same pattern shows up in the
geometry-type mix — MGCP captures most small features as points rather
than polygons at this scale.

In [ ]:
# Sanity check: are we picking up everything in the cell that's a polygon?

# How many shapefiles by prefix
print("Shapefiles in cell by prefix:")
for prefix in ["A", "P", "L", "T"]:
    n = sum(1 for p in MGCP_DIR.glob(f"{prefix}*.shp"))
    print(f"  {prefix}: {n}")

# Total features per geometry type across the whole cell
print("\nFeatures across the cell by geometry-type prefix:")
for prefix, label in [("A", "polygons"), ("P", "points"), ("L", "lines")]:
    total = 0
    for shp in MGCP_DIR.glob(f"{prefix}*.shp"):
        total += len(gpd.read_file(shp))
    print(f"  {prefix} ({label}): {total:,}")

The points total is meaningfully larger than the polygons total — a hint
that the bulk of MGCP's feature coverage in this cell is captured as
points, not polygons. Matching points to Overture polygons is a different
methodology (point-in-polygon) and out of scope for this demo, but it's
worth knowing the asymmetry is there.

## Schema comparison: MGCP buildings vs. Overture buildings

Before matching, it's worth understanding what each side's building schema
actually contains. For this comparison we focus on the
`MGCP AL015 (Building)` ↔ `Overture buildings/building` pair — the most
central case in the demo. The same principles apply to the other passes,
but with different specific attributes.

The point of the comparison isn't to translate attributes between schemas.
In this methodology, attributes stay on whichever side they came from —
the matching pipeline only produces a link between identifiers. The
comparison is to help orient anyone who's going to do something with
the matched results downstream.

### MGCP AL015 in this dataset

AL015 has 47 attribute fields in the TRD 3.0 spec. The table below shows
fields that are populated in this cell's 412 building records, sorted from
universally populated to sparse. Many spec-defined fields are present in
the data but contain MGCP's "no information" sentinels (`998`, `-32767`,
`'UNK'`, `'N_A'`) on every record — those are not shown.

Attribute descriptions come from NGA's
[Geospatial Analysis Integrity Tool (GAIT)](https://github.com/ngageoint/Geospatial-Analysis-Integrity-Tool),
which encodes the canonical MGCP TRD 3.0 attribute definitions in
[`mgcp3_attr.c`](https://github.com/ngageoint/Geospatial-Analysis-Integrity-Tool/blob/master/GAIT%2026%20Source/mgcp3_attr.c).

In [ ]:
# Attribute descriptions sourced from NGA's GAIT tool, which encodes the
# authoritative MGCP TRD 3.0 attribute definitions:
# https://github.com/ngageoint/Geospatial-Analysis-Integrity-Tool/blob/master/GAIT%2026%20Source/mgcp3_attr.c
mgcp_buildings_schema = pd.DataFrame([
    ("UID",        "100%", "MGCP Feature universally unique identifier (UUID per ISO/IEC 9834-8)"),
    ("FCODE",      "100%", "Feature code (always 'AL015' for buildings in this shapefile)"),
    ("FUN",        "100%", "Functional Use — the general categories of function or use that a facility or site serves"),
    ("WID",        "100%", "Width perpendicular to the feature's primary alignment, in meters"),
    ("ACC",        "100%", "Horizontal Accuracy Category — a general categorical evaluation of horizontal accuracy"),
    ("ACE",        "100%", "Absolute Horizontal Accuracy — circular error at 90% probability, in meters"),
    ("SRC_DATE",   "100%", "Source Date and Time — when the source data was collected"),
    ("SRC_INFO",   "100%", "Source Description — a description of the data set used"),
    ("CPYRT_NOTE", "100%", "Commercial Copyright Notice"),
    ("TIER_NOTE",  "100%", "Commercial Distribution Restriction"),
    ("CIT",         "28%", "Corrections Facility Type"),
    ("PPO",         "22%", "Product — principal product(s) of a production, mining, or agricultural activity"),
    ("DDC",         "21%", "Dwelling Type"),
    ("ICF",         "14%", "Manufacturing Facility Type"),
    ("PAF",         "14%", "Public Accommodation Facility Type"),
    ("TXT",         "11%", "Associated Text — narrative or other textual description"),
    ("NAM",          "6%", "Name"),
    ("HGT",          "1%", "Height Above Surface Level — base-to-top height in meters"),
], columns=["Field", "Populated", "Description"])
mgcp_buildings_schema

### Overture `buildings/building`

Overture's building schema is documented at
[docs.overturemaps.org](https://docs.overturemaps.org/schema/reference/buildings/building/).
The table below shows the fields most relevant for downstream integration
work. (`subtype` and `class` form a two-level taxonomy of building purpose;
the rest cover physical and contextual attributes.)

In [ ]:
# Field descriptions sourced from Overture's schema docs:
# https://docs.overturemaps.org/schema/reference/buildings/building/
overture_buildings_schema = pd.DataFrame([
    ("id",                     "A feature ID, which may be associated with the Global Entity Reference System (GERS) if the feature is part of GERS"),
    ("bbox",                   "An optional bounding box for the feature"),
    ("geometry",               "The building's footprint or roofprint, traced from aerial/satellite imagery"),
    ("theme",                  "Always 'buildings'"),
    ("type",                   "Always 'building'"),
    ("version",                "Feature version"),
    ("sources",                "Information about the source data used to assemble the feature"),
    ("subtype",                "A broad classification of the current use and purpose of the building"),
    ("class",                  "A more specific classification of the current use and purpose"),
    ("has_parts",              "Whether the building has associated building part features"),
    ("names",                  "Building names"),
    ("level",                  "Z-order of the feature where 0 is visual level"),
    ("height",                 "Height of the building or part in meters (lowest point to highest)"),
    ("is_underground",         "Whether the entire building is completely below ground"),
    ("num_floors",             "Number of above-ground floors"),
    ("num_floors_underground", "Number of below-ground floors"),
    ("min_height",             "Altitude above ground where the bottom of the building starts (for floating buildings)"),
    ("min_floor",              "Start floor of this building (for floating buildings)"),
    ("facade_color",           "Facade color in hex notation"),
    ("facade_material",        "Outer surface material of the facade"),
    ("roof_material",          "Outer surface material of the roof"),
    ("roof_shape",             "Shape of the roof"),
    ("roof_direction",         "Bearing of the roof ridge line in degrees"),
    ("roof_orientation",       "Orientation of the roof shape relative to the footprint shape"),
    ("roof_color",             "Roof color in hex notation"),
    ("roof_height",            "Height of the roof in meters (base of the roof to its highest point)"),
], columns=["Field", "Description"])
overture_buildings_schema

### What the comparison tells us

A few observations from looking at both sides:

- **Both schemas have a stable identifier** that's intended to persist
  across edits and releases. MGCP's `UID` and Overture's `id` (a GERS ID)
  are both UUIDs. The matching pipeline produces a join between these two
  identifiers; that's this notebook's central deliverable.

- **The shape of attribute coverage differs.** Overture's schema is broader
  (more attribute fields available), while MGCP is more sparsely populated
  in practice (many spec fields exist but contain "no information"
  sentinels). For an attribute that matters to a downstream user — say,
  height — Overture is more likely to have a value populated, but MGCP's
  provenance and accuracy metadata is more complete.

- **Some attributes don't have direct counterparts.** MGCP's `ACE`
  (positional accuracy) and `TIER_NOTE` (release restrictions) have no
  Overture analog. Overture's `roof_shape` and `facade_material` have no
  MGCP analog. This is fine — in this methodology, those attributes stay
  on whichever side captured them, and they're available for whoever holds
  that side of the data to use as they see fit.

A full attribute mapping (deciding how `MGCP FUN=2` translates to an
Overture `subtype`/`class` pair, for instance) is a separate exercise that
isn't required for the matching to work. The match table links the
identifiers; the attributes ride along.

## Matching methodology

For each pair of polygons (one from MGCP, one from an Overture type), we
compute two quantities:

1. **Intersection over Union (IoU)** — the ratio of the area of intersection
   to the area of union. Ranges from 0 (no overlap) to 1 (identical
   footprints). IoU is the standard polygon-matching metric: it rewards
   overlap while penalizing size mismatch.

2. **Centroid containment** — whether the centroid of the Overture polygon
   falls inside the MGCP polygon. This is asymmetric on purpose. It catches
   the case where the MGCP polygon is significantly larger than the Overture
   one (a single MGCP capture covering what Overture has split into multiple
   smaller footprints) and the IoU alone wouldn't be high enough to declare
   a match.

A pair is considered a match if it meets one of two criteria:

| Tier | Condition |
| --- | --- |
| High | `IoU >= 0.5` |
| Low  | `IoU >= 0.3` *and* the Overture centroid is contained in the MGCP polygon |

This two-tier rule is more permissive than a flat IoU threshold and is
better-suited to matching across datasets with different capture scales,
where one side may have aggregated multiple structures into a single
polygon. Pairs that meet neither criterion are recorded as
non-matches by virtue of their absence from the output table.

### Reproject to a metric CRS

IoU and centroid-containment both depend on geometric operations that need
consistent units. The raw data on both sides is in `EPSG:4326` (longitude
and latitude in degrees), which means a degree of longitude is about 111 km
at the equator and shrinks toward the poles. We can't meaningfully compute
areas or distances in degrees.

We reproject both sides to `EPSG:32617` (UTM Zone 17N), which covers the
demo tile and uses meters as its unit. For matching pipelines covering
different geographies, you'd pick a different UTM zone, or use a global
equal-area projection like `EPSG:6933`.

In [ ]:
METRIC_CRS = "EPSG:32617"
mgcp_m = mgcp.to_crs(METRIC_CRS)
overture_m = {key: gdf.to_crs(METRIC_CRS) for key, gdf in overture.items()}
print(f"MGCP: {len(mgcp_m):,} features in {METRIC_CRS}")
for key, gdf in overture_m.items():
    print(f"  {key:<30} {len(gdf):>7,} features in {METRIC_CRS}")

### Single-pass matching function

This function takes the MGCP polygons and one Overture GeoDataFrame, runs
the spatial join to find candidate pairs, applies the two-tier rule, and
returns a long-form table of accepted matches.

In [ ]:
def match_pass(mgcp_gdf, overture_gdf, pass_name):
    """Run one matching pass: MGCP polygons against one Overture type.

    Returns a DataFrame with one row per matched pair, including IoU and
    tier information. Pairs that don't meet either tier's criterion are
    not included.
    """
    if len(overture_gdf) == 0:
        return pd.DataFrame(columns=[
            "mgcp_uid", "mgcp_fcode", "overture_id", "pass_name",
            "iou", "centroid_contained", "tier",
        ])

    # Spatial join: every (MGCP, Overture) pair whose geometries intersect.
    # GeoPandas uses a spatial index internally, so this is fast even when
    # one side has tens of thousands of features.
    joined = gpd.sjoin(
        mgcp_gdf[["UID", "FCODE", "geometry"]],
        overture_gdf[["id", "geometry"]],
        predicate="intersects",
        how="inner",
        lsuffix="mgcp",
        rsuffix="ovt",
    )
    if len(joined) == 0:
        return pd.DataFrame(columns=[
            "mgcp_uid", "mgcp_fcode", "overture_id", "pass_name",
            "iou", "centroid_contained", "tier",
        ])

    # sjoin keeps the left geometry. We need both — pull the right geometry
    # back in by joining on the index sjoin recorded.
    mgcp_geom = joined.geometry.reset_index(drop=True)
    ovt_geom = overture_gdf.geometry.iloc[joined["index_ovt"].values].reset_index(drop=True)

    # Vectorized IoU and centroid-containment.
    inter_area = mgcp_geom.intersection(ovt_geom, align=False).area
    union_area = mgcp_geom.union(ovt_geom, align=False).area
    iou = inter_area / union_area.where(union_area > 0)
    centroid_in = ovt_geom.centroid.within(mgcp_geom, align=False)

    # Apply the two-tier rule
    high = iou >= 0.5
    low = (iou >= 0.3) & centroid_in
    keep = (high | low).values

    out = pd.DataFrame({
        "mgcp_uid":           joined["UID"].values[keep],
        "mgcp_fcode":         joined["FCODE"].values[keep],
        "overture_id":        joined["id"].values[keep],
        "pass_name":          pass_name,
        "iou":                iou.values[keep],
        "centroid_contained": centroid_in.values[keep],
        "tier":               ["high" if h else "low" for h in high.values[keep]],
    })
    return out.reset_index(drop=True)

### Run all seven passes

One matching pass per Overture type, with the results concatenated into a
single long-form table. We run the passes in a deliberate order: the
cleanest case first (buildings), then adjacent themes with similar
behavior, then the cross-schema friction cases (`base/land` and
`base/land_cover`) where MGCP's specific codes meet Overture's broader
taxonomy. A given MGCP feature can appear in multiple passes if it
matched something in more than one Overture type.

In [ ]:
all_matches = []
for key, ovt_gdf in overture_m.items():
    print(f"Matching {key}...", end=" ", flush=True)
    pass_matches = match_pass(mgcp_m, ovt_gdf, pass_name=key)
    print(f"{len(pass_matches):>5} matches")
    all_matches.append(pass_matches)
matches = pd.concat(all_matches, ignore_index=True)
print(f"\nTotal matches across all passes: {len(matches):,}")
matches.head(10)

### A first look at the matches

A quick summary of what we have:

In [ ]:
print(f"Total matched pairs:     {len(matches):,}")
print(f"Distinct MGCP features:  {matches['mgcp_uid'].nunique():,} (out of {len(mgcp_m):,})")
print(f"Distinct Overture ids:   {matches['overture_id'].nunique():,}")
print()
print("Matches per pass:")
per_pass = matches.groupby("pass_name", as_index=False).agg(
    n_matches=("mgcp_uid", "size"),
    n_high_tier=("tier", lambda s: (s == "high").sum()),
    n_low_tier=("tier", lambda s: (s == "low").sum()),
).sort_values("n_matches", ascending=False)
per_pass

## Cardinality reporting

A matched pair on its own doesn't tell us much about how to integrate two
datasets. What matters is the *pattern* of matches: does each MGCP polygon
correspond cleanly to one Overture feature, or does it overlap many smaller
ones? Are there MGCP features that match nothing at all? Are there Overture
features in the cell that nothing in MGCP captured?

We classify each MGCP polygon by its match cardinality, separately for each
pass. The categories are:

- **1:1** — one MGCP polygon, exactly one Overture match. The clean case.
- **1:many** — one MGCP polygon, several Overture features inside or
   overlapping it. Typical when MGCP has aggregated at 1:100K what Overture
   captured as individual features.
- **many:1** — multiple MGCP polygons matched the same Overture feature.
   Rarer; usually indicates fragmentation on the MGCP side.
- **1:0** — MGCP has the feature, Overture doesn't (in this pass).
- **0:1** — Overture has a feature in this category, MGCP doesn't. Tracked
   separately since it's not a property of any MGCP UID.

The 1:0 and 0:1 cases are the most interpretively loaded. Whether a 0:1
means "MGCP missed it" or "MGCP correctly didn't capture it at this scale"
is the kind of question the cardinality table surfaces but doesn't answer
on its own; that judgment requires domain knowledge about extraction policy.


In [ ]:
def classify_cardinality_per_pass(matches, mgcp_gdf, overture_dict):
    """Per (MGCP UID, pass) classification of match cardinality.

    Returns a long-form DataFrame with one row per (mgcp_uid, pass_name)
    combination — including passes where the MGCP polygon found no match
    (1:0 case). The 0:1 case (Overture features with no MGCP match) is
    summarized separately by the caller.
    """
    rows = []
    pass_names = list(overture_dict.keys())

    # Count matches per (mgcp_uid, pass_name)
    per_pass_counts = (
        matches.groupby(["mgcp_uid", "pass_name"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=pass_names, fill_value=0)
    )

    # For each pass, also count how many MGCP polygons matched each Overture id.
    # This is what lets us distinguish 1:1 from many:1.
    overture_match_counts = {
        pass_name: matches[matches["pass_name"] == pass_name]
        .groupby("overture_id")["mgcp_uid"]
        .nunique()
        .to_dict()
        for pass_name in pass_names
    }

    # All MGCP UIDs, including ones with no matches in any pass
    all_uids = mgcp_gdf["UID"].unique()

    for uid in all_uids:
        for pass_name in pass_names:
            n_overture_matched = (
                per_pass_counts.loc[uid, pass_name]
                if uid in per_pass_counts.index else 0
            )
            if n_overture_matched == 0:
                cardinality = "1:0"
            else:
                # Are any of the matched Overture features also matched by
                # other MGCP polygons in this same pass?
                matched_ovt_ids = matches[
                    (matches["mgcp_uid"] == uid)
                    & (matches["pass_name"] == pass_name)
                ]["overture_id"].tolist()
                any_shared = any(
                    overture_match_counts[pass_name].get(oid, 0) > 1
                    for oid in matched_ovt_ids
                )
                if n_overture_matched == 1 and not any_shared:
                    cardinality = "1:1"
                elif n_overture_matched > 1 and not any_shared:
                    cardinality = "1:many"
                else:
                    cardinality = "many:1"
            rows.append({
                "mgcp_uid": uid,
                "pass_name": pass_name,
                "n_overture_matched": n_overture_matched,
                "cardinality": cardinality,
            })
    return pd.DataFrame(rows)


per_pass_cardinality = classify_cardinality_per_pass(matches, mgcp_m, overture_m)
print(f"Cardinality rows: {len(per_pass_cardinality):,}")
print(f"({len(mgcp_m):,} MGCP polygons × {len(overture_m)} passes)")
per_pass_cardinality.head(10)

### Per-pass cardinality breakdown

How does the cardinality distribute across passes? The shape of this table
is the diagnostic — different passes have very different signatures, and
those differences drive different integration choices downstream.

In [ ]:
per_pass_breakdown = (
    per_pass_cardinality
    .groupby(["pass_name", "cardinality"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["1:1", "1:many", "many:1", "1:0"], fill_value=0)
)

# Also add the 0:1 count per pass (Overture features unmatched by any MGCP)
matched_ovt_ids_by_pass = (
    matches.groupby("pass_name")["overture_id"].apply(set).to_dict()
)
zero_to_one_counts = {
    pass_name: len(set(gdf["id"]) - matched_ovt_ids_by_pass.get(pass_name, set()))
    for pass_name, gdf in overture_m.items()
}
per_pass_breakdown["0:1"] = pd.Series(zero_to_one_counts)

per_pass_breakdown = per_pass_breakdown.reindex(list(overture_m.keys()))
per_pass_breakdown

### Global cardinality summary

The per-pass breakdown shows how each MGCP polygon relates to one Overture
type at a time. But a single MGCP polygon can match features across
*multiple* passes — an AL015 building footprint might match a building in
`buildings/building` *and* a residential zone in `base/land_use`. The
global view asks a different question: across all seven passes combined,
how does each MGCP polygon land?

We classify each MGCP polygon by:

- **Total Overture matches** across all passes
- **Number of distinct passes** it matched in
- **A global cardinality label**, which collapses the per-pass labels into
  a single per-polygon summary

In [ ]:
def classify_cardinality_global(per_pass_cardinality, mgcp_gdf):
    """One row per MGCP polygon, summarizing matches across all passes.

    The global cardinality label rolls up the per-pass labels:
    - 'unmatched': 1:0 in every pass
    - 'clean': all matched passes are 1:1
    - 'fragmented': at least one many:1 pass
    - 'aggregated': at least one 1:many pass, and no many:1
    - 'mixed': both 1:many and many:1 patterns appear across passes
    """
    grouped = per_pass_cardinality.groupby("mgcp_uid")
    summary = grouped.agg(
        n_overture_matched=("n_overture_matched", "sum"),
        n_passes_matched=("n_overture_matched", lambda s: (s > 0).sum()),
        cardinality_set=("cardinality", lambda s: set(s)),
    ).reset_index()

    def label(cset):
        non_zero = cset - {"1:0"}
        if not non_zero:
            return "unmatched"
        has_one_many = "1:many" in non_zero
        has_many_one = "many:1" in non_zero
        if has_one_many and has_many_one:
            return "mixed"
        if has_many_one:
            return "fragmented"
        if has_one_many:
            return "aggregated"
        return "clean"

    summary["global_cardinality"] = summary["cardinality_set"].apply(label)
    summary = summary.drop(columns=["cardinality_set"])

    fcodes = mgcp_gdf[["UID", "FCODE"]].rename(columns={"UID": "mgcp_uid", "FCODE": "mgcp_fcode"})
    summary = summary.merge(fcodes, on="mgcp_uid", how="left")

    return summary[["mgcp_uid", "mgcp_fcode", "n_passes_matched", "n_overture_matched", "global_cardinality"]]


global_cardinality = classify_cardinality_global(per_pass_cardinality, mgcp_m)
print(f"Global cardinality rows: {len(global_cardinality):,}")
global_cardinality.head(10)

### Top-line distribution

How do the 1,348 MGCP polygons distribute across the five global categories?

In [ ]:
global_breakdown = (
    global_cardinality["global_cardinality"]
    .value_counts()
    .reindex(["clean", "aggregated", "fragmented", "mixed", "unmatched"])
    .to_frame("polygons")
)
global_breakdown["pct"] = (global_breakdown["polygons"] / len(global_cardinality) * 100).round(1)
global_breakdown

### How many passes does a typical matched polygon land in?

A clean 1:1 building match might land in only `buildings/building`; an
AL015 footprint inside a built-up residential zone might land in
`buildings/building` *and* `base/land_use`. The distribution of
pass-counts tells us whether passes are mostly independent or whether
multi-pass matches are common.

In [ ]:
pass_count_distribution = (
    global_cardinality[global_cardinality["n_passes_matched"] > 0]
    ["n_passes_matched"]
    .value_counts()
    .sort_index()
    .to_frame("polygons")
)
pass_count_distribution["pct of matched"] = (
    pass_count_distribution["polygons"] / pass_count_distribution["polygons"].sum() * 100
).round(1)
pass_count_distribution

### Cardinality by MGCP feature code

The global breakdown tells us how the 1,348 polygons distribute across
cardinality classes. But the polygons aren't a homogeneous group — they're
34 different feature codes with very different match behaviors. AL015
buildings match cleanly to Overture buildings. EB020 thickets fragment
against Overture's broader land cover. BA030 islands sometimes match
base/land cleanly and sometimes split across multiple Overture features.

For the GERS adoption decision matrix in the next section, the relevant
unit is the feature code, not the dataset as a whole. So we cross-tabulate.

In [ ]:
fcode_cardinality = (
    global_cardinality
    .groupby(["mgcp_fcode", "global_cardinality"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["clean", "aggregated", "fragmented", "mixed", "unmatched"], fill_value=0)
)
fcode_cardinality["total"] = fcode_cardinality.sum(axis=1)
fcode_cardinality = fcode_cardinality.sort_values("total", ascending=False)

# Join the feature-code names from the lookup so the table is human-readable
fcode_cardinality = (
    fcode_cardinality
    .merge(fcode_names[["feature_code", "name"]], left_index=True, right_on="feature_code", how="left")
    .set_index("feature_code")
)
fcode_cardinality = fcode_cardinality[["name", "clean", "aggregated", "fragmented", "mixed", "unmatched", "total"]]
fcode_cardinality

A few patterns worth calling out, reading top-down (highest-volume codes first):

- **AL015 (Building):** the cleanest signal in the dataset. The vast
  majority of matched buildings land in the `clean` bucket, with a small
  tail where one Overture building was the best match for more than one
  MGCP footprint.
- **EB020 (Thicket) and EC030 (Trees):** dominated by `unmatched` or
  `fragmented`. These codes are the friction story — MGCP has fine-grained
  vegetation differentiation that Overture's broader land_cover taxonomy
  doesn't preserve.
- **BA030 (Island) and ED010 (Marsh):** mixed behavior, often `fragmented`.
  Both are large-extent features that Overture captures with different
  boundaries.
- **The single-feature codes** (AT050, AM040, AD030, etc.): often just one
  row each. The methodology behaves the same; the sample is too small to
  draw any pattern from.

Different feature codes will need different GERS adoption strategies. The
next section turns this table into a decision framework.

### Which feature codes match cleanly, and which don't?

A simpler view of the same data: for each feature code, what fraction of
its polygons matched at all, and of those matches, what fraction were clean?

In [ ]:
fcode_match_rates = pd.DataFrame({
    "name": fcode_cardinality["name"],
    "total": fcode_cardinality["total"],
    "matched": fcode_cardinality["total"] - fcode_cardinality["unmatched"],
    "clean_matches": fcode_cardinality["clean"],
})
fcode_match_rates["match_rate"] = (
    fcode_match_rates["matched"] / fcode_match_rates["total"] * 100
).round(1)
fcode_match_rates["clean_rate"] = (
    fcode_match_rates["clean_matches"] / fcode_match_rates["matched"].where(fcode_match_rates["matched"] > 0) * 100
).round(1)
fcode_match_rates = fcode_match_rates[["name", "total", "matched", "match_rate", "clean_matches", "clean_rate"]]
fcode_match_rates

The match_rate column tells us "how many polygons of this feature code found any Overture counterpart" — i.e., the inverse of the unmatched rate. The clean_rate column, computed only over matched polygons, tells us "of the ones that matched, how often was it a clean 1:1 in every pass" — i.e., the quality of the matches we did find.
Two different feature codes with the same match rate can tell very different stories: high match rate with high clean rate means GERS adoption is straightforward; high match rate with low clean rate means GERS adoption needs a link table.

## Next steps

The methodology so far produces:

- A match table (`matches`) — one row per matched MGCP/Overture pair, with
  IoU and tier.
- A global cardinality classification per MGCP polygon.
- A per-feature-code diagnostic showing where direct GERS adoption works
  and where a link table is needed.

Sections still to come in this notebook:

- **GERS adoption decision matrix.** Synthesize the per-fcode diagnostic
  into concrete adoption paths (direct ID attachment, link table, deferred).
- **PMTiles publication.** Visualize the matches with one or two Overture
  attributes joined on for inspection. Approach TBD.
- **Wrap-up.** Summary, limitations, and what running this against
  operational TDS v7 or current MGCP TRD 4 data would change.

See the companion lesson `7-buildings-matching.md` for the prose
walkthrough.